# ECMWF Open Data (IFS forecasts): quickstart

A minimal end-to-end pull of a real-time IFS forecast field from ECMWF
Open Data, opened with xarray via cfgrib, and plotted as a global map.

ECMWF publishes a subset of its operational IFS output as free open data
under CC-BY-4.0. The data is global at 0.25 degree resolution, with four
forecast runs per day (00z, 06z, 12z, 18z). Only the most recent 2-3 days
of runs are available (rolling archive, no history).

Full reference: [`docs/ecmwf-open-data/README.md`](../docs/ecmwf-open-data/README.md).

## Before you run this

No registration or API key is needed. Just install the Python stack:

```bash
pip install ecmwf-opendata cfgrib eccodes xarray matplotlib cartopy
```

The default config below fetches the latest deterministic forecast of
2 metre temperature at T+24 h: one GRIB2 file, typically a few MB, back
in under a minute. Edit the config cell to change variables, steps, or
forecast type.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
# GRIB short names. See docs/ecmwf-open-data/README.md for common variables.
PARAMS = ["2t"]

# Forecast lead time in hours.
STEP = 24

# Forecast type: "fc" (HRES deterministic), "cf" (control), "pf" (perturbed)
TYPE = "fc"

# Forecast stream: "oper" (00z/12z), "scda" (06z/18z), "enfo" (ensemble)
STREAM = "oper"

# Model: "ifs", "aifs-single", "aifs-ens"
MODEL = "ifs"

# Where to save the output.
OUTPUT_DIR = "../data/ecmwf-open-data"
OUTPUT_FILENAME = "ecmwf_open_data_quickstart.grib2"
# ==================================================================

## Imports and environment check

The cell below prints the version of every library used in this notebook.
Unlike CDS-based datasets, ECMWF Open Data needs no credentials, so there
is no credential check. The repo root is located so that shared utilities
can be imported if needed.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python           {sys.version.split()[0]}")
for pkg in ["ecmwf-opendata", "cfgrib", "eccodes", "xarray", "matplotlib", "cartopy"]:
    print(f"{pkg:16} {version(pkg)}")

# Find the repo root by walking up from CWD until we see CLAUDE.md.
# This works whether Jupyter was launched from the repo root or from
# the notebooks/ directory.
def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError(
        "Could not find repo root. Run this notebook from inside the "
        "climate-data-quickstart repository."
    )

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"\nRepo root: {repo_root}")

## Download a forecast field

The `ecmwf-opendata` client fetches the latest available run by default.
With the config above this retrieves 2 metre temperature at T+24 h from
the most recent 00z or 12z deterministic (HRES) run. The file is GRIB2
format, typically a few MB for a single global field.

In [ ]:
from ecmwf.opendata import Client

output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
output_path.parent.mkdir(parents=True, exist_ok=True)

client = Client(source="ecmwf", model=MODEL, stream=STREAM)
client.retrieve(
    step=STEP,
    type=TYPE,
    param=PARAMS,
    target=str(output_path),
)

print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

xarray opens the GRIB2 file via the cfgrib engine. The variable will
appear under its GRIB short name (`t2m` for 2 metre temperature, since
cfgrib rewrites `2t` to avoid leading digits). Check the coordinates for
the forecast reference time (`time`) and valid time (`valid_time` or
`step`). Values are in kelvin.

In [ ]:
ds = xr.open_dataset(str(output_path), engine="cfgrib")
print(ds)

## Plot a global map

A Robinson projection is a good default for global forecast data. The
temperature is converted from kelvin to Celsius before plotting. The
colourbar is labelled with units, and coastlines provide geographic
context.

In [ ]:
# Identify the temperature variable (cfgrib rewrites "2t" to "t2m")
t2m_var = "t2m" if "t2m" in ds.data_vars else list(ds.data_vars)[0]
t2m = ds[t2m_var].squeeze()
t2m_celsius = t2m - 273.15

fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.Robinson())

t2m_celsius.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "2 m temperature (C)", "shrink": 0.6},
)

ax.coastlines(resolution="110m", linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
ax.set_global()

# Build a descriptive title from dataset coordinates
ref_time = str(ds.coords["time"].values)[:16] if "time" in ds.coords else "unknown"
step_val = int(ds.coords["step"].values / 1e9 / 3600) if "step" in ds.coords else STEP
ax.set_title(f"IFS forecast: 2 m temperature, init {ref_time} UTC, T+{step_val} h")

plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/ecmwf-open-data/README.md`](../docs/ecmwf-open-data/README.md)
  for variables, access parameters, gotchas, and licence details.
- **Multiple variables**: add more short names to the `PARAMS` list (e.g.
  `["2t", "msl", "10u", "10v"]`). Each additional parameter adds a few MB.
- **Multiple lead times**: loop over steps to build a forecast time series
  for a grid point (e.g. `for step in range(0, 72, 3)`). Each step is a
  separate retrieve call.
- **Ensemble forecasts**: set `STREAM = "enfo"` and `TYPE = "cf"` or
  `TYPE = "pf"` to fetch control or perturbed ensemble members.
- **AIFS model**: set `MODEL = "aifs-single"` to fetch ECMWF's AI-based
  forecast instead of the physics-based IFS.
- **Compare with reanalysis**: see the `era5-single-levels` entry in this
  repo for the historical counterpart to these forecasts.